In [ ]:
!pip install catboost

In [ ]:
import kagglehub
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import seaborn as sns
from catboost import CatBoostClassifier
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
missing_frac = df.isna().mean().sort_values(ascending=False)
missing_frac = missing_frac[missing_frac > 0]

print(f"Columns with missing values: {len(missing_frac)}")

if len(missing_frac) > 0:
    missing_table = (missing_frac * 100).round(2).to_frame('missing_%')
    display(missing_table.head(150)) if 'display' in globals() else print(missing_table.head(20))

    plt.figure(figsize=(10,4))
    missing_table.head(20).plot(kind='bar', legend=False)
    plt.title('Missing % (top 20)')
    plt.ylabel('%')
    plt.tight_layout()
    plt.show()
    print(missing_table[missing_frac > 0.5])
    dropped = missing_table[missing_frac > 0.5]
    df.drop(dropped.index , axis=1)
else:
    print("No missing values detected.")
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object','category','bool']).columns.tolist()

# Numeric: median
if len(num_cols) > 0:
    df[num_cols] = df[num_cols].fillna(df[num_cols].median(numeric_only=True))

# Categorical: unknown
if len(cat_cols) > 0:
    df[cat_cols] = df[cat_cols].fillna('unknown')

print('Remaining missing values:', int(df.isna().sum().sum()))


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Task 2,3,4,5: Write your code here:
# The softmax function
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot
from tqdm import tqdm
def softmax(z):
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss
model = CatBoostClassifier(verbose=0,
    n_estimators=200,
    max_depth=4)
def gradient_descent(X, y, num_classes, lr, n_iters=1000):
  # Get the number of samples (m) and number of features (n)
  m, n = X.shape

  # Initialize weight matrix with shape (n, num_classes)
  theta = np.zeros((n, num_classes))

  # One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)

  losses = []

  for _ in tqdm(range(n_iters), desc="Training Multiclass Logistic Regression"):
    # Calculate the logits z
    z = np.dot(X, theta)

    # Get class probabilities using softmax
    y_pred = softmax(z)

    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m

    # Update weights
    theta -= lr * gradient

    # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)
  return theta, losses

n_splits = 5 # K=5 Folds
#stratified becuase of the imbalance
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

# Get the train & test split for this fold
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Train using gradient descent with learning rate = 0.5
theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

# Calculate z & class probabilities for X_test
z = np.dot(X_test, theta)
y_pred_proba = softmax(z)

# Pick the predicted classes with the highest probability
y_pred = np.argmax(y_pred_proba, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

# Store results
sr_results['loss'].append(losses)
sr_results['acc'].append(accuracy)
sr_results['f1'].append(f1)
#TODO: Calculate the average losses across folds
avg_loss = np.mean(sr_results['loss'], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from matplotlib.figure import Figure
importances = {}


imp = model.feature_importances_

# Create a 1x3 plot
fig = Figure(figsize=(10, 6))
features = X.columns

sorted_idx = np.argsort(imp)
fig.barh(features[sorted_idx], imp[sorted_idx])
fig.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: